<div class="alert alert-block alert-success">

# Molecular simulations for drug discovery and design

### A Lorentz Center – CECAM workshop · Bogotá, Colombia · 23–27 November 2026

## Hands-on session: Molecular Dynamics and Enhanced Sampling

</div>

**Molecular dynamics (MD)** is a computational technique that simulates the time evolution of particles by numerically solving Newton's equations of motion.

MD shows how molecular systems behave at the atomistic level: their structure, flexibility and interactions. In drug discovery it is used to study the dynamics of proteins, folding and conformational changes, and the binding of potential drugs to a protein receptor.

In this notebook, we build an MD simulation from scratch on a simple toy model, and then use it to understand **enhanced sampling** and free energy calculations.

**Part A: Molecular Dynamics**
1. A particle in a double-well potential
2. Integrating Newton's equations of motion with Velocity Verlet (NVE ensemble)
3. Adding a thermostat (NVT ensemble)
4. From sampling to free energies: the Boltzmann distribution

**Part B: Enhanced Sampling**

5. The rare event problem
6. Setting up Umbrella Sampling
7. Running Umbrella Sampling simulations
8. Reconstructing the Potential of Mean Force (PMF) with WHAM

<div class="alert alert-block alert-warning">

**Running on Google Colab**

1. Go to **File → Save a copy in Drive**, so that you work on your own copy.
2. Run the first code cell: it installs RDKit (about 30 seconds). All other packages are already available in Colab.
3. If Colab disconnects (for example after a long pause), your code is kept but the variables are lost: use **Runtime → Run all** to get back to where you were.
</div>

<div class="alert alert-block alert-info">

**How this session works**

- Fill in the code where you see `### START MODIFICATION ###` or `### ADD YOUR CODE HERE ###`.
- Answer the questions in the **blue boxes** and discuss them.
- Run the cells **in order**. If something breaks, restart the runtime and run all cells from the top.

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

# Part A: Molecular Dynamics

When studying the time evolution of proteins or other biomolecules, researchers rely on MD engines such as GROMACS, NAMD, AMBER or OpenMM. To understand how these programs work, we will implement their fundamental algorithms ourselves.

For simplicity, we focus on a 1D toy model: a particle moving in a double-well potential.

We will first integrate Newton's equations of motion in the microcanonical **NVE** ensemble, where the total energy is conserved. Then we will add a thermostat (the Andersen thermostat) to simulate the canonical **NVT** ensemble, where the temperature is constant. This is the relevant ensemble for biomolecular systems, since most biological processes occur at constant temperature.

<h2 style="color:green;"> Step 1: A particle moving in a double-well potential </h2>

We consider a particle moving in the 1D double-well potential

$$V(x) = a\,(x^2 - b^2)^2$$

with the corresponding force

$$F(x) = -\frac{dV}{dx} = -4a\,x\,(x^2 - b^2)$$

In [ ]:
# Define the double-well potential
def double_well_potential(x, a=1.0, b=1.0):
    ''' V(x) = a * (x^2 - b^2)^2 '''
    return a * (x**2 - b**2)**2

# Define the corresponding force
def double_well_force(x, a=1.0, b=1.0):
    ''' F(x) = -dV/dx '''
    return -4*a*x*(x**2 - b**2)

# Plot the double-well potential
x_vals = np.linspace(-2, 2, 200)

plt.figure(figsize=(8,6))
plt.plot(x_vals, double_well_potential(x_vals), 'k-')
plt.xlabel("x")
plt.ylabel("V(x)")
plt.title("1D double-well potential")
plt.ylim(-0.2, 3)
plt.show()

The two wells (at $x = \pm b$) represent stable states. The barrier between them, of height $V(0) = a\,b^4$, is the energy the particle must overcome to move from one well to the other.

Now change the parameter $a$ to modify the barrier height, and plot the different potentials.

In [ ]:
### CREATE A LIST OF a VALUES TO MODIFY THE BARRIER HEIGHT ###
### START MODIFICATION ###
a_list =
### END MODIFICATION ###

plt.figure(figsize=(8,6))
for a_val in a_list:
    plt.plot(x_vals, double_well_potential(x_vals, a=a_val), label=f"a = {a_val}")

plt.xlabel("x")
plt.ylabel("V(x)")
plt.title("1D double-well potential")
plt.ylim(-0.2, 6)
plt.legend()
plt.show()

<div class="alert alert-block alert-info">

**Question 1.** What happens when you increase $\alpha$?
Do you expect the transitions between the wells to become less or more frequent?
</div>

From here on, we use $a = b = 1$, so the barrier height is $\Delta V = 1$.

<h2 style="color:green;"> Step 2: Integrating Newton's equations of motion with Velocity Verlet </h2>

To obtain the trajectory of the system, we need to numerically integrate Newton's equations of motion. Several algorithms exist; here we use **Velocity Verlet**, the standard choice in MD thanks to its simplicity, stability and time-reversibility.

Given the position $x(t)$, velocity $v(t)$ and force $F(t)$ at time $t$, one step of length $\Delta t$ is:

1. Update the position:
$$x(t+\Delta t) = x(t) + v(t)\,\Delta t + \frac{F(t)}{2m}\,\Delta t^2$$

2. Compute the force at the new position:
$$F(t+\Delta t) = F\big(x(t+\Delta t)\big)$$

3. Update the velocity using the average of the old and new forces:
$$v(t+\Delta t) = v(t) + \frac{F(t) + F(t+\Delta t)}{2m}\,\Delta t$$

Complete the following function (where indicated) with the position and velocity updates.

In [ ]:
# --- Velocity Verlet integrator (NVE) ---

def velocity_verlet(x0, v0, m, dt, n_steps):
    '''
    Integrate the motion of a particle using Velocity Verlet.

    Parameters:
        x0, v0 : initial position and velocity
        m      : mass
        dt     : time step
        n_steps: number of integration steps
    Returns:
        t_traj, x_traj, v_traj : arrays of times, positions and velocities
                                 (index 0 is the initial state)
    '''
    x = x0
    v = v0
    F = double_well_force(x)

    t_traj = np.zeros(n_steps + 1)
    x_traj = np.zeros(n_steps + 1)
    v_traj = np.zeros(n_steps + 1)
    x_traj[0], v_traj[0] = x, v

    for i in range(n_steps):
        # 1. Update position
        ### START MODIFICATION FOR POSITION ###
        x +=
        ### END MODIFICATION FOR POSITION ###
        # 2. Compute new force
        F_new = double_well_force(x)
        # 3. Update velocity
        ### START MODIFICATION FOR VELOCITY ###
        v += 
        ### END MODIFICATION FOR VELOCITY ###
        # 4. Update force for next step
        F = F_new

        # Save trajectory
        t_traj[i+1] = (i+1)*dt
        x_traj[i+1] = x
        v_traj[i+1] = v

    return t_traj, x_traj, v_traj

In [ ]:
def plot_trajectory_and_energy(t, x, v, m, title):
    E_tot = 0.5*m*v**2 + double_well_potential(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
    ax1.plot(t, x, 'b-')
    ax1.set_xlabel("Time")
    ax1.set_ylabel("x")
    ax1.set_title(f"Trajectory ({title})")
    ax2.plot(t, E_tot, 'r-')
    ax2.set_xlabel("Time")
    ax2.set_ylabel("Total energy")
    ax2.set_title(f"Total energy ({title})")
    ax2.set_ylim(E_tot.min()-0.1, E_tot.max()+0.1)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Simulation parameters ---
x0 = -1.0         # start in the left well
v0 = 0.0          # initial velocity
m = 1.0           # mass
dt = 0.01         # time step
n_steps = 5000    # number of integration steps

# --- Run NVE simulation ---
t_nve, x_nve, v_nve = velocity_verlet(x0, v0, m, dt, n_steps)
plot_trajectory_and_energy(t_nve, x_nve, v_nve, m, "NVE")

This is a simulation in the NVE ensemble: there is no energy exchange with the environment, so the total energy is conserved.

With an initial velocity of zero (`v0 = 0`), the particle sits at the minimum $x = -1$ and does not move. 

Give it some kinetic energy by setting a non-zero initial velocity.

In [ ]:
# --- Simulation parameters ---
x0 = -1.0
v0 = 1           # change the initial velocity
m = 1.0
dt = 0.01
n_steps = 5000

# --- Run NVE simulation ---
t_nve, x_nve, v_nve = velocity_verlet(x0, v0, m, dt, n_steps)
plot_trajectory_and_energy(t_nve, x_nve, v_nve, m, "NVE")

The total energy is still conserved; small fluctuations around the mean are normal for a finite time step.

With a small initial velocity, the particle oscillates around $x = -1$. When the initial velocity is large enough, it crosses the barrier and reaches the other minimum at $x = +1$.

<div class="alert alert-block alert-info">

**Question 2.** What is the **smallest initial velocity** for which the particle crosses the barrier?

First find it by trial and error, then derive it from energy conservation (the barrier height is $\Delta V = a\,b^4 = 1$).

(*Hint: compare the particle's initial kinetic energy $\tfrac{1}{2} m v_0^2$ with the barrier height.*)
</div>

<h2 style="color:green;"> Step 3: Adding a thermostat (NVT ensemble) - Andersen thermostat </h2>

To simulate a system at **constant temperature**, the particle must exchange energy with a thermal bath. This is the canonical (NVT) ensemble: the temperature is fixed, while the total energy fluctuates.

This is done by adding a **thermostat**. One simple approach is the Andersen thermostat:

- At each time step, the particle has a probability $\nu\,\times \Delta t$ of having its velocity reassigned at random from the Maxwell-Boltzmann distribution at the target temperature $T$ ($\nu$ is the collision frequency).

- Between these "collisions", the system evolves with Velocity Verlet exactly as in NVE.

The Maxwell-Boltzmann distribution in 1D is a normal distribution centred at $0$ with standard deviation $\sigma = \sqrt{k_B T / m}$. We work in reduced units, with $k_B = 1$.

Complete the function below. The Velocity Verlet updates are the same as Step 2, but now the new velocity is drawn from the Maxwell-Boltzmann distribution (hint: use `np.random.normal`).

In [ ]:
# --- Velocity Verlet integrator with Andersen thermostat (NVT) ---

def velocity_verlet_NVT(x0, v0, m, dt, n_steps, T=1.0, nu=1.0, force=double_well_force):
    '''
    Integrate the motion of a particle using Velocity Verlet with the Andersen thermostat.

    Parameters:
        x0, v0 : initial position and velocity
        m      : mass
        dt     : time step
        n_steps: number of integration steps
        T      : temperature (reduced units)
        nu     : collision frequency of the Andersen thermostat
        force  : function returning the force F(x)
    Returns:
        t_traj, x_traj, v_traj : arrays of times, positions and velocities
    '''
    kB = 1.0  # Boltzmann constant in reduced units

    x = x0
    v = v0
    F = force(x)

    t_traj = np.zeros(n_steps + 1)
    x_traj = np.zeros(n_steps + 1)
    v_traj = np.zeros(n_steps + 1)
    x_traj[0], v_traj[0] = x, v

    for i in range(n_steps):
        # --- Velocity Verlet steps ---
        # 1. Update position
        ### START MODIFICATION FOR POSITION ###
        x += 
        ### END MODIFICATION FOR POSITION ###
        # 2. Compute new force
        F_new = force(x)
        # 3. Update velocity
        ### START MODIFICATION FOR VELOCITY ###
        v += 
        ### END MODIFICATION FOR VELOCITY ###
        # 4. Update force for next step
        F = F_new

        # --- Andersen thermostat ---
        if np.random.rand() < nu * dt:
            # Reassign velocity from the Maxwell-Boltzmann distribution
            ### START MODIFICATION ###
            v = 
            ### END MODIFICATION ###

        # Store trajectory
        t_traj[i+1] = (i+1)*dt
        x_traj[i+1] = x
        v_traj[i+1] = v

    return t_traj, x_traj, v_traj

Now, also draw the **initial velocity** of the simulation from the Maxwell-Boltzmann distribution.

In [ ]:
# --- Simulation parameters ---
x0 = -1.0          # start in the left well
m = 1.0
dt = 0.01
n_steps = 100000   # total time = n_steps * dt = 1000
T = 0.3            # reduced temperature
nu = 1.0           # collision frequency
kB = 1.0           # Boltzmann constant in reduced units

# Initialize the velocity from the Maxwell-Boltzmann distribution
### ADD YOUR CODE HERE ###
v0 = 

# --- Run NVT simulation ---
t_nvt, x_nvt, v_nvt = velocity_verlet_NVT(x0, v0, m, dt, n_steps, T, nu)
plot_trajectory_and_energy(t_nvt, x_nvt, v_nvt, m, f"NVT, T = {T}")

<div class="alert alert-block alert-info">

**Question 3.** Is the total energy conserved in the NVT simulation? Why?
</div>

In the NVE simulation, the particle either always crosses or never crosses, depending on its energy. In NVT, the energy fluctuates, and the particle crosses the barrier only when a thermal fluctuation happens to give it enough energy. Crossing is therefore a **random event**.

Let's measure how often it happens. The function below counts the transitions between the two wells (a transition is counted when the particle reaches $x > 0.5$ coming from the left well, or $x < -0.5$ coming from the right well).

In [ ]:
def count_transitions(x_traj, threshold=0.5):
    ''' Count the transitions between the two wells along a trajectory. '''
    state = -1 if x_traj[0] < 0 else 1
    n_transitions = 0
    for x in x_traj:
        if state == -1 and x > threshold:
            state, n_transitions = 1, n_transitions + 1
        elif state == 1 and x < -threshold:
            state, n_transitions = -1, n_transitions + 1
    return n_transitions

Run NVT simulations at several temperatures (same length, total time 1000) and count the transitions.

In [ ]:
T_list = [0.1, 0.15, 0.2, 0.3, 0.5]

fig, axes = plt.subplots(len(T_list), 1, figsize=(10, 2.2*len(T_list)), sharex=True)
for ax, T_i in zip(axes, T_list):
    v0 = np.random.normal(0, np.sqrt(kB*T_i/m))
    t, x, v = velocity_verlet_NVT(-1.0, v0, m, dt, n_steps, T_i, nu)
    n_tr = count_transitions(x)
    ax.plot(t, x, 'b-', lw=0.5)
    ax.set_ylabel("x")
    ax.set_title(f"T = {T_i}: {n_tr} transitions", fontsize=10)
axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">

**Question 4.**

1. How does the number of transitions change with temperature?

2. The average waiting time between transitions follows approximately the Arrhenius law, $\tau \propto e^{\Delta V / k_B T}$. With $\Delta V = 1$, how much longer do you have to wait at $T = 0.1$ than at $T = 0.5$?

3. How often do you expect the transition to be at a given temperature if the barrier $\Delta V$ increases?
</div>

<h2 style="color:green;"> Step 4: From sampling to free energies - the Boltzmann distribution </h2>

A thermostat does more than keep the temperature constant: it makes the simulation sample the **canonical (Boltzmann) distribution**. The probability of finding the particle at position $x$ is

$$\rho(x) = \frac{e^{-V(x)/k_B T}}{Z}, \qquad Z = \int e^{-V(x)/k_B T}\,dx$$

The PMF can be determined from the sampled distribution:

$$\mathrm{F}(x) = -k_B T \ln \rho(x) + \text{const}$$

This relation is the basis of free energy calculations from simulations, and of Part B. Let's check it with a long NVT simulation at $T = 0.5$, where the particle crosses the barrier often.

In [ ]:
T_check = 0.5
v0 = np.random.normal(0, np.sqrt(kB*T_check/m))
t, x_long, v = velocity_verlet_NVT(-1.0, v0, m, dt, 500000, T_check, nu)

# Histogram of the sampled positions
bins = np.linspace(-2, 2, 81)
rho, edges = np.histogram(x_long, bins=bins, density=True)
x_centers = 0.5*(edges[1:] + edges[:-1])

# Exact Boltzmann distribution
boltzmann = np.exp(-double_well_potential(x_vals)/(kB*T_check))
boltzmann /= np.sum(boltzmann)*(x_vals[1] - x_vals[0])   # normalize

# PMF from the histogram
with np.errstate(divide='ignore'):
    pmf = -kB*T_check*np.log(rho)
pmf -= np.min(pmf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
ax1.bar(x_centers, rho, width=edges[1]-edges[0], alpha=0.5, label="Sampled (NVT)")
ax1.plot(x_vals, boltzmann, 'k-', label="Exact Boltzmann")
ax1.set_xlabel("x")
ax1.set_ylabel(r"$\rho(x)$")
ax1.legend()

ax2.plot(x_centers, pmf, 'bo', label=r"$-k_BT\ln\rho(x)$ from simulation")
ax2.plot(x_vals, double_well_potential(x_vals), 'k-', label="V(x)")
ax2.set_xlabel("x")
ax2.set_ylabel("PMF")
ax2.set_ylim(-0.2, 2.5)
ax2.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">

**Question 5.** Where does the PMF from the simulation agree with $V(x)$ (Figure 1), and where is it noisy or missing? Why? What happens with the same simulation at $T = 0.1$?
</div>

<div class="alert alert-block alert-warning">
Simulations in the NVT ensemble are important because most biological and chemical processes occur at constant temperature, not constant energy.

A thermostat ensures that the system samples configurations according to the canonical ensemble. This is what allows us to compute properties like free energies, probabilities and reaction rates from a simulation, provided that **all relevant states are sampled**.
</div>

# Part B: Enhanced Sampling

Today, MD can simulate processes on the microsecond to millisecond timescale. However, many processes of interest in biochemistry and pharmaceutical chemistry, such as protein folding and ligand binding and unbinding, happen on much longer timescales. In simulations, these processes are called **rare events**: as we saw in Question 4, the waiting time grows exponentially with the barrier height.

Conventional MD is too slow to sample rare events. **Enhanced sampling methods** accelerate them and allow us to explore the free energy landscape efficiently. Here, we focus on one of the most widely used: **Umbrella Sampling (US)**.

<h2 style="color:green;"> Step 5: The rare event problem </h2>

In a real protein-ligand system, there are thousands to millions of degrees of freedom (protein, ligand, solvent, ions...), so the free energy landscape is a $3N$-dimensional hypersurface. Since it is impossible to visualize, we describe the free energy along a **reaction coordinate** that captures the most important motion of the process. This one-dimensional projection of the free energy landscape is the **Potential of Mean Force (PMF)**.

We consider again a particle in a 1D double well, now **asymmetric**. Even though it is a very simplified model, it captures the essential features of protein-ligand binding:

- The lower minimum is the most stable state, analogous to the **bound state** (ligand in the binding pocket).
- The higher minimum is analogous to the **unbound state**. (In a real system the unbound state is a flat region rather than a well, but the analogy is enough here.)
- The barrier between them mimics the activation free energy between bound and unbound states.

In [ ]:
# Parameters of the asymmetric double well
a_us = 4.0   # barrier height
b_us = 1.0   # position of the minima
c_us = 1.0   # tilt parameter: makes the left minimum lower

def asym_potential(x):
    ''' Asymmetric double well: V(x) = a (x^2 - b^2)^2 + c x '''
    return a_us*(x**2 - b_us**2)**2 + c_us*x

def asym_force(x):
    ''' F(x) = -dV/dx '''
    return -(4*a_us*x*(x**2 - b_us**2) + c_us)

# "True" PMF, shifted so that its minimum is zero
x_grid = np.linspace(-1.6, 1.5, 500)
V_min = np.min(asym_potential(x_grid))
true_PMF = asym_potential(x_grid) - V_min

plt.figure(figsize=(8,5))
plt.plot(x_grid, true_PMF, 'k-', lw=2)
plt.xlabel('x')
plt.ylabel('PMF')
plt.title('Asymmetric double well')
plt.show()

<div class="alert alert-block alert-warning">
<b>Note:</b> In this toy model, the PMF is known analytically, so we can compare our results with the exact answer. In a real protein-ligand system, the free energy landscape is unknown: we do not know in advance the positions of the minima and barriers, nor the free energy difference. Computing it is the goal of the simulation.
</div>

**Binding free energy calculations** often start from an X-ray structure of the protein-ligand complex. Let's do the equivalent here: start the particle in the lower ("bound") minimum and run a standard NVT simulation at $T = 0.5$, using the integrator you wrote in Part A. At this temperature the barrier seen from the lower minimum is about $10\,k_BT$.

In [ ]:
T_us = 0.5     # reduced temperature for Part B
dt_us = 0.001  # time step for Part B

v0 = np.random.normal(0, np.sqrt(kB*T_us/m))
t_unb, x_unb, v_unb = velocity_verlet_NVT(-1.0, v0, m, dt_us, 2000000, T=T_us, nu=1.0,
                                          force=asym_force)

# PMF from the unbiased histogram
bins = np.linspace(-1.6, 1.5, 63)
rho, edges = np.histogram(x_unb, bins=bins, density=True)
x_centers = 0.5*(edges[1:] + edges[:-1])
with np.errstate(divide='ignore'):
    pmf_unb = -kB*T_us*np.log(rho)
pmf_unb -= np.min(pmf_unb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
ax1.plot(t_unb[::100], x_unb[::100], 'b-', lw=0.5)
ax1.set_xlabel("Time")
ax1.set_ylabel("x")
ax1.set_title(f"Unbiased MD: {count_transitions(x_unb)} transitions")
ax2.plot(x_grid, true_PMF, 'k-', lw=2, label="True PMF")
ax2.plot(x_centers, pmf_unb, 'bo', label="Unbiased MD")
ax2.set_xlabel("x")
ax2.set_ylabel("PMF")
ax2.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">

**Question 6.** How many times does the particle cross the barrier? Can you reliably estimate the barrier height and the free energy difference between the two minima from this simulation?
</div>

<h2 style="color:green;"> Step 6: Setting up Umbrella Sampling </h2>

### Reaction coordinate

A **reaction coordinate** (RC) describes the progress of a process. It reduces the high-dimensional motion of all atoms to a single variable along which we compute the free energy.

In a real system, defining a good reaction coordinate is challenging. We use **collective variables** (CVs): functions of the atomic coordinates that capture the slowest or most important motions. They can be simple, such as the distance between the ligand and the binding site or a torsion angle. They can also be more advanced, such as path collective variables (progress along a predefined path) or machine-learning-based CVs.

The choice of CV is critical: it must capture the essential physics of the process for enhanced sampling to work. In our 1D model, the reaction coordinate is simply the position of the particle, $x$.

### Umbrella windows

In **umbrella sampling**, an additional harmonic **bias potential** is applied along the CV $\xi$:

$$V_{\rm US}(x) = \frac{1}{2} k \left(\xi(x)-\xi_0\right)^2$$

where $k$ is the force constant and $\xi_0$ is the center of the restraint. Here $\xi = x$.

Multiple simulations ("windows") are run, each with the restraint centred at a different $\xi_0$. Each window forces the system to sample a different region of the CV, including high free energy regions such as the barrier.

Setting up US requires choosing two key parameters:
- the **centers** of the umbrella windows, $\xi_0$
- the **force constant**, $k$

### Window centers

In real systems, the relevant range of the CV is not known in advance, and preliminary simulations may be needed to place the windows.

<div class="alert alert-block alert-warning"><b>Note:</b> Adjacent windows must have sufficient <b>overlap</b> in the sampled values of the CV. Adequate overlap is essential for a reliable reconstruction of the PMF.</div>

Here, we use equally spaced windows with $\Delta x = 0.25$, from $x = -1.5$ to $x = 1.5$ (13 windows).

In [ ]:
# Umbrella sampling window centers
delta_x = 0.5
centers = np.arange(-1.5, 1.5 + delta_x, delta_x)
print("Window centers:", centers)

### Force constant

The choice of the force constant $k$ is also crucial:
- If $k$ is **too small**, the bias is too weak to hold the system on the barrier: it slides back into the minima.
- If $k$ is **too large**, the system stays tightly confined around $\xi_0$, and adjacent windows do not overlap.

The plot below shows the **total** potential felt by the particle in each window, $V(x) + V_{\rm US}(x)$. Try different values of $k$.

In [ ]:
# Force constant of the harmonic bias (used everywhere below)
k_us = 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
for x0_us in centers:
    V_bias = 0.5*k_us*(x_grid - x0_us)**2
    ax1.plot(x_grid, V_bias, label=f'center = {x0_us:.1f}')
    ax2.plot(x_grid, true_PMF + V_bias, label=f'center = {x0_us:.1f}')
ax1.set_title(f'Bias potentials, k = {k_us}')
ax1.set_ylim(0, 7)
ax2.plot(x_grid, true_PMF, 'k--', lw=2, label='V(x)')
ax2.set_title(f'Total potential V(x) + bias, k = {k_us}')
ax2.set_ylim(0, 12)
for ax in (ax1, ax2):
    ax.set_xlabel('x')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">

**Question 7.**

1. What happens to the total potential of the window centred at $x = 0$ (on the barrier) when $k$ is small, e.g. $k = 5$?

2. The curvature of the PMF at the top of the barrier is $V''(0) = -4ab^2 = -16$. What is the minimum value of $k$ needed for the total potential of this window to have a single minimum?
</div>

<h2 style="color:green;"> Step 7: Running Umbrella Sampling simulations </h2>

In unbiased MD, the force is $F(x) = -dV/dx$. In umbrella sampling, the particle feels the total potential

$$V_{\rm tot}(x) = V(x) + V_{\rm US}(x), \qquad V_{\rm US}(x) = \frac{1}{2} k (x-x_0)^2$$

so the total force is

$$F_{\rm tot}(x) = F(x) + F_{\rm US}(x), \qquad F_{\rm US}(x) = -\frac{dV_{\rm US}}{dx} = -k\,(x-x_0)$$

Complete the umbrella force below. We then simply pass the total force to the NVT integrator from Part A: nothing else changes!

In [ ]:
# -------- Umbrella force --------
def force_umbrella(x, k, x0_us):
    ''' Umbrella force: -k (x - x0) '''
    ### START MODIFICATION ###
    F_us = 
    ### END MODIFICATION ###
    return F_us


# -------- Run one simulation per window --------
def run_umbrella_sampling(centers, k, n_steps):
    ''' Run one biased NVT simulation per window and return the trajectories. '''
    trajectories = np.zeros((len(centers), n_steps + 1))
    for i, x0_us in enumerate(centers):
        print(f"Running window {i+1}/{len(centers)} centred at x = {x0_us:.2f}")
        total_force = lambda x: asym_force(x) + force_umbrella(x, k, x0_us)
        t, x, v = velocity_verlet_NVT(
            x0=x0_us,              # start the simulation at the center of the window
            v0=0.0,
            m=1.0,
            dt=dt_us,
            n_steps=n_steps,
            T=T_us,
            nu=1.0,
            force=total_force,
        )
        trajectories[i] = x
    return trajectories


def plot_window_histograms(trajectories, centers, k):
    plt.figure(figsize=(9,5))
    for i, x0_us in enumerate(centers):
        plt.hist(trajectories[i], bins=40, density=True, alpha=0.5, label=f"center = {x0_us:.1f}")
    plt.xlabel("x")
    plt.ylabel("Biased probability density")
    plt.title(f"Biased histograms from the umbrella windows (k = {k})")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

First, a short run with the value of `k_us` you chose above, to check the overlap between windows (this takes a few seconds).

In [ ]:
trajectories = run_umbrella_sampling(centers, k_us, n_steps=200000)
plot_window_histograms(trajectories, centers, k_us)

<div class="alert alert-block alert-info">

**Question 7.**

1. Is there enough overlap between adjacent windows?

2. Change `k_us` (in the cell of Step 6) and re-run the short simulations. Which range of $k$ gives a good overlap, while still sampling the barrier region?
</div>

Once you have chosen a good value of $k$ (hint: try $k = 20$), run the production simulations. They are longer, for better statistics, and take about a minute.

In [ ]:
k_us = 20                 # final choice of the force constant
n_steps_us = 1000000      # the longer, the better the sampling

trajectories = run_umbrella_sampling(centers, k_us, n_steps=n_steps_us)
plot_window_histograms(trajectories, centers, k_us)

<h2 style="color:green;"> Step 8: Reconstructing the Potential of Mean Force (PMF) </h2>

As we saw in Step 4, the PMF is obtained from the Boltzmann distribution:

$$\mathrm{PMF}(x) = -k_B T \ln \rho(x)$$

However, the histograms from umbrella sampling are **biased**: each window sampled $x$ according to $V(x) + V_{\rm US}(x)$, not $V(x)$. The biased distribution of window $i$ is

$$\rho_i^{\rm biased}(x) \propto \rho(x)\, e^{-V_{{\rm US},i}(x)/k_BT}$$

### First attempt: unbiasing each window separately

Since the bias is known, we can remove it window by window:

$$\mathrm{PMF}(x) = -k_B T \ln \rho_i^{\rm biased}(x) - V_{{\rm US},i}(x) + f_i$$

where $f_i$ is an **unknown constant** that is different for every window.

### WHAM: the Weighted Histogram Analysis Method

**WHAM** finds the constants $f_i$ and the best estimate of $\rho(x)$ by combining all windows in a statistically optimal way. It solves two equations self-consistently, by iterating until convergence:

$$\rho(x) = \frac{\sum_i n_i(x)}{\sum_i N_i\, e^{-\left[V_{{\rm US},i}(x) - f_i\right]/k_BT}}
\qquad\qquad
e^{-f_i/k_BT} = \sum_x \rho(x)\, e^{-V_{{\rm US},i}(x)/k_BT}$$

where $n_i(x)$ is the histogram count of window $i$ in bin $x$, and $N_i$ is the total number of samples in window $i$.

Programs such as `wham` (Grossfield lab), `gmx wham` (GROMACS) or the Python package `pymbar` implement this method. For a 1D problem, it takes only a few lines of numpy.

In [ ]:
def wham(trajectories, centers, k, kT, bins, tol=1e-6, max_iter=100000):
    '''
    1D Weighted Histogram Analysis Method.

    Parameters:
        trajectories : array (n_windows, n_samples) of sampled CV values
        centers      : umbrella centers x0_i
        k            : umbrella force constant
        kT           : thermal energy k_B*T
        bins         : bin edges for the histograms
    Returns:
        x_centers : bin centers
        pmf       : PMF(x), shifted so that its minimum is zero
        f         : free energy constants f_i of the windows
    '''
    x_centers = 0.5*(bins[1:] + bins[:-1])

    # Histogram counts n_i(x) and number of samples N_i of each window
    n_ix = np.array([np.histogram(traj, bins=bins)[0] for traj in trajectories])
    N_i = n_ix.sum(axis=1)

    # Bias potential of each window evaluated at each bin center
    V_bias = 0.5*k*(x_centers[None, :] - centers[:, None])**2

    f = np.zeros(len(centers))
    for iteration in range(max_iter):
        # Equation 1: unbiased probability rho(x)
        denominator = np.sum(N_i[:, None]*np.exp(-(V_bias - f[:, None])/kT), axis=0)
        rho = n_ix.sum(axis=0)/denominator
        # Equation 2: free energy constants f_i
        f_new = -kT*np.log(np.sum(rho[None, :]*np.exp(-V_bias/kT), axis=1))
        f_new -= f_new[0]  # only differences matter
        if np.max(np.abs(f_new - f)) < tol:
            break
        f = f_new
    print(f"WHAM converged in {iteration+1} iterations")

    with np.errstate(divide='ignore'):
        pmf = -kT*np.log(rho)
    pmf -= np.min(pmf)
    return x_centers, pmf, f

In [ ]:
x_wham, pmf_wham, f_wham = wham(trajectories, centers, k_us, kT, bins)

plt.figure(figsize=(8,5))
plt.plot(x_grid, true_PMF, 'k-', lw=2, label='True PMF')
plt.plot(x_wham, pmf_wham, 'bo', ms=5, label='Umbrella sampling + WHAM')
plt.plot(x_centers, pmf_unb, 'r^', ms=4, alpha=0.6, label='Unbiased MD (Step 5)')
plt.xlabel('x', fontsize=15)
plt.ylabel('PMF', fontsize=15)
plt.legend()
plt.grid()
plt.show()

If adjacent windows overlap well and the sampling is long enough, the PMF from umbrella sampling matches the true PMF, including the barrier and the higher minimum that unbiased MD could not sample. (Small differences come from incomplete sampling; they shrink if you increase `n_steps_us`.)

Finally, we can read off the quantities of interest.

In [ ]:
# Locate the two minima and the barrier on the WHAM PMF
left = x_wham < 0
x_min_left = x_wham[left][np.argmin(pmf_wham[left])]
x_min_right = x_wham[~left][np.argmin(pmf_wham[~left])]
between = (x_wham > x_min_left) & (x_wham < x_min_right)
barrier = np.max(pmf_wham[between])
dG = np.min(pmf_wham[~left]) - np.min(pmf_wham[left])

# Same quantities from the exact potential
true_left = true_PMF[x_grid < 0].min()
true_right = true_PMF[x_grid > 0].min()
true_barrier = true_PMF[(x_grid > -0.9) & (x_grid < 0.9)].max()

print(f"Free energy difference (upper - lower minimum): WHAM = {dG:.2f}, exact = {true_right - true_left:.2f}")
print(f"Barrier from the lower minimum:                 WHAM = {barrier:.2f}, exact = {true_barrier - true_left:.2f}")

<div class="alert alert-block alert-info">

**Question 8.**

1. How well do the WHAM free energy difference and barrier height agree with the exact values?

2. Repeat the reconstruction with short simulations at $k = 200$ (poor overlap). What happens to the PMF?

3. Repeat the reconstruction with fewer windows (e.g. delta_x = 1). What happens to the PMF?
</div>

<div class="alert alert-block alert-warning">

In real systems, unlike this toy model, the shape of the free energy landscape is not known in advance. Enhanced sampling methods such as umbrella sampling are therefore essential to estimate the PMF of biological processes, such as protein-ligand binding.

From the PMF along a binding coordinate, one can compute the **binding free energy** of a protein-ligand complex. This is crucial in drug discovery, as it reflects the affinity of a ligand, and hence of a potential drug, for its protein target.

</div>